# Day 16 - Schema Evolution and Schema Drift

**Topic:** Schema Evolution and Schema Drift  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** เข้าใจความต่างระหว่าง `schema drift` กับ `schema evolution` และฝึก handle schema change แบบควบคุมได้ใน Lakehouse table

วันนี้เราจะจำลองสถานการณ์ที่ source schema เปลี่ยนระหว่าง batch เช่น มี column ใหม่, column หาย, หรือ data type เปลี่ยน แล้วฝึกคิดว่า case ไหนควร evolve target table, case ไหนควรปรับ incoming DataFrame ให้ตรงกับ target table schema, และ case ไหนควร flag/reject ก่อนเขียนลง Lakehouse table

## Goal of the day

1. แยกให้ออกว่า **schema drift** และ **schema evolution** ต่างกันอย่างไร
2. อธิบายได้ว่า source schema เปลี่ยนแบบไหนที่ควร accept, reject หรือ investigate
3. ใช้ PySpark ตรวจ new columns, missing columns และ type mismatch เบื้องต้นได้
4. เขียนข้อมูลลง Lakehouse table โดยใช้ controlled schema evolution อย่างระวังได้
5. เริ่มสร้าง habit ว่าก่อน append/merge ต้องตรวจ schema ไม่ใช่เชื่อ source ทันที

## Why it matters in production

ใน production pipeline schema เปลี่ยนได้เสมอ เช่น source เพิ่ม field ใหม่, ลบ field เดิม, rename column, หรือส่ง type มาไม่เหมือนเดิม

ถ้า pipeline ไม่จัดการเรื่องนี้ให้ดี อาจเกิดปัญหาเช่น:

- append เข้า table ไม่ผ่าน เพราะ schema mismatch
- column ใหม่หายไปเงียบ ๆ เพราะไม่ได้ evolve schema
- missing column ทำให้ downstream report ได้ค่า null หรือ error
- type mismatch ทำให้ cast ผิด, data หาย หรือ metric เพี้ยน
- pipeline accept schema change อัตโนมัติเกินไปจน table schema บวมและควบคุมยาก

Production takeaway:

> Schema drift ต้องถูกมองเห็นก่อน ส่วน schema evolution ต้องเกิดแบบตั้งใจ ไม่ใช่ปล่อยให้ source เปลี่ยน table ตามใจเอง

## Key concepts

| Concept | Meaning |
|---|---|
| Schema | โครงสร้างของ DataFrame/Table เช่น column name, data type, nullable |
| Schema Drift | Source schema เปลี่ยนไปจากที่ pipeline คาดไว้ เช่น column ใหม่, column หาย, type เปลี่ยน |
| Schema Evolution | การเปลี่ยน target table schema แบบตั้งใจและควบคุมได้ เช่น เพิ่ม column ใหม่ที่ยอมให้ค่าเป็น null ได้ |
| Schema Mismatch | Source กับ target schema ไม่ตรงกัน ทำให้ write/read/transform อาจ fail |
| New Column | Column ใหม่จาก source ที่ target ยังไม่มี ต้องตัดสินใจว่าจะ ignore, log หรือ evolve |
| Missing Column | Column ที่ target คาดหวังแต่ source batch ไม่มี ต้องเติม null หรือ fail ตาม business rule |
| Type Mismatch | Column เดิมแต่ data type เปลี่ยน เช่น `int` กลายเป็น `string` ต้อง cast/validate อย่างระวัง |
| Schema Contract | ข้อตกลงว่า table นี้ควรมี columns และ types อะไร ไม่ใช่ปล่อยให้เปลี่ยนไร้ทิศทาง |

## Concept explanation

### Schema Drift คืออะไร?

**Schema drift** คือการที่ schema ของ incoming data เปลี่ยนไปจากที่ pipeline เคยคาดไว้ เช่น:

- source เพิ่ม `phone_number`
- source ไม่ส่ง `email` มาแล้ว
- source ส่ง `customer_score` จาก `integer` กลายเป็น `string`
- source เปลี่ยนชื่อ column จาก `status` เป็น `customer_status`

พูดง่าย ๆ:

> Drift = สิ่งที่ source เปลี่ยนเข้ามา  
> Evolution = สิ่งที่เราอนุญาตให้ target table เปลี่ยนตามอย่างมี control

### Schema Evolution ควรใช้เมื่อไร?

Schema evolution เหมาะกับการเปลี่ยนแปลงที่ปลอดภัยและตั้งใจ เช่น เพิ่ม optional column ใหม่ที่ downstream ยังไม่จำเป็นต้องใช้ทันที

ตัวอย่างที่พอรับได้:

```text
เพิ่ม column: loyalty_tier STRING
เพิ่ม column: phone_number STRING
```

ตัวอย่างที่ควรระวังมากกว่า:

```text
ลบ column: email
เปลี่ยน type: customer_score INT -> STRING
rename column: status -> customer_status
```

### Controlled schema change mindset

เวลามี schema drift ไม่ควรรีบเขียนลง table ทันที แต่ควรถามก่อนว่า:

1. Column ใหม่นี้จำเป็นไหม?
2. Missing column เป็นเรื่องปกติของ source batch นี้หรือเป็น source issue?
3. Type mismatch cast ได้ปลอดภัยไหม?
4. Downstream query/report จะได้รับผลกระทบไหม?
5. เราควร log schema drift นี้ไว้ตรวจสอบหรือไม่?

สำหรับ learning lab วันนี้ เราจะฝึก 3 pattern หลัก:

- detect schema difference
- evolve schema เฉพาะกรณี column ใหม่ที่ยอมรับได้
- align missing columns ให้ตรงกับ target schema และ handle type mismatch ก่อน append


## Mermaid diagram: Schema Drift Handling Flow

```mermaid
flowchart LR
    A[Incoming source batch] --> B[Compare with target schema]
    B --> C{Drift type?}
    C -- New optional columns --> D[Controlled schema evolution]
    C -- Missing expected columns --> E[Add missing columns as null or fail]
    C -- Type mismatch --> F[Cast, validate, or reject]
    C -- Unknown/unsafe change --> G[Stop and investigate]
    D --> H[Append to Lakehouse table]
    E --> H
    F --> I{Valid after cast?}
    I -- Yes --> H
    I -- No --> J[Flag or reject records]
```

Key idea:

> ไม่ใช่ schema drift ทุกแบบที่ควร evolve target table ตาม source ทันที

## Setup / imports

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, 4faee51e-54ad-44c2-ae6c-7482b450a9ef, 3, Finished, Available, Finished, False)

## Create mock data

เราจะเริ่มจาก source batch แรกที่เป็น schema version 1 แล้วเขียนเป็น Lakehouse table ชื่อ `day16_customer_profile_schema_demo`

In [2]:
target_table_name = "day16_customer_profile_schema_demo"

customer_schema_v1 = T.StructType([
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("email", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("customer_score", T.IntegerType(), True),
    T.StructField("updated_at", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
])

customer_batch_v1_data = [
    ("C001", "Alice Wong", "alice@example.com", "active", 85, "2026-05-01 09:00:00", "crm"),
    ("C002", "Boon Mee", "boon@example.com", "active", 72, "2026-05-01 09:10:00", "crm"),
    ("C003", "Chai Lee", "chai@example.com", "inactive", 60, "2026-05-01 09:20:00", "crm"),
]

df_customer_v1 = (
    spark.createDataFrame(customer_batch_v1_data, customer_schema_v1)
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

df_customer_v1.show(truncate=False)
df_customer_v1.printSchema()

StatementMeta(, 4faee51e-54ad-44c2-ae6c-7482b450a9ef, 4, Finished, Available, Finished, False)

+-----------+-------------+-----------------+--------+--------------+-------------------+-------------+
|customer_id|customer_name|email            |status  |customer_score|updated_at         |source_system|
+-----------+-------------+-----------------+--------+--------------+-------------------+-------------+
|C001       |Alice Wong   |alice@example.com|active  |85            |2026-05-01 09:00:00|crm          |
|C002       |Boon Mee     |boon@example.com |active  |72            |2026-05-01 09:10:00|crm          |
|C003       |Chai Lee     |chai@example.com |inactive|60            |2026-05-01 09:20:00|crm          |
+-----------+-------------+-----------------+--------+--------------+-------------------+-------------+

root
 |-- customer_id: string (nullable = false)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- status: string (nullable = true)
 |-- customer_score: integer (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- sourc

## PySpark code examples

ใน section นี้จะจำลอง schema drift หลายแบบ แล้วค่อย ๆ handle ด้วย PySpark และ Lakehouse table operation

### Example 1: Create the initial Lakehouse table

เริ่มจากการเขียน batch แรกเป็น target table ด้วย schema version 1

เราใช้ `overwriteSchema=true` เพราะเป็น demo table และต้องการ reset table ทุกครั้งที่ run notebook จากต้นจนจบ

In [3]:
(
    df_customer_v1.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table_name)
)

print(f"Created table: {target_table_name}")
spark.read.table(target_table_name).printSchema()

StatementMeta(, 4faee51e-54ad-44c2-ae6c-7482b450a9ef, 5, Finished, Available, Finished, False)

Created table: day16_customer_profile_schema_demo
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- status: string (nullable = true)
 |-- customer_score: integer (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- source_system: string (nullable = true)



### Example 2: New columns from source

Batch ถัดมามี column ใหม่คือ:

- `phone_number`
- `loyalty_tier`

นี่คือ schema drift แบบ **new columns**

โดยปกติไม่ควร assume ว่า append แล้ว target table จะ evolve เองเสมอ เราจะลอง append แบบ strict ก่อน แล้วค่อยใช้ `mergeSchema` แบบตั้งใจ

In [4]:
customer_schema_v2_new_columns = T.StructType([
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("email", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("customer_score", T.IntegerType(), True),
    T.StructField("updated_at", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("phone_number", T.StringType(), True),
    T.StructField("loyalty_tier", T.StringType(), True),
])

customer_batch_v2_data = [
    ("C004", "Dao Jai", "dao@example.com", "active", 91, "2026-05-02 10:00:00", "crm", "0811111111", "gold"),
    ("C005", "Ekkarat Tan", "ekkarat@example.com", "active", 77, "2026-05-02 10:05:00", "crm", "0822222222", "silver"),
]

df_customer_v2_new_columns = (
    spark.createDataFrame(customer_batch_v2_data, customer_schema_v2_new_columns)
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

df_customer_v2_new_columns.printSchema()

StatementMeta(, 4faee51e-54ad-44c2-ae6c-7482b450a9ef, 6, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = false)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- status: string (nullable = true)
 |-- customer_score: integer (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)



### Example 3: Try strict append before schema evolution

Cell นี้ตั้งใจให้เห็นว่า append แบบไม่เปิด schema evolution ควร fail ถ้า target table ยังไม่มี column ใหม่

ถ้า append ผ่านแบบไม่ได้ตั้งใจ ให้ตรวจสอบว่า Spark session หรือ runtime configuration มีการเปิด schema evolution ไว้หรือไม่ เช่น `spark.databricks.delta.schema.autoMerge.enabled`

In [5]:
append_without_merge_succeeded = False

try:
    (
        df_customer_v2_new_columns.write
        .format("delta")
        .mode("append")
        .saveAsTable(target_table_name)
    )
    append_without_merge_succeeded = True
    print("Append without mergeSchema succeeded.")
    print("Check whether schema evolution is enabled at the Spark session or runtime configuration level.")
except Exception as e:
    print("Expected append issue because incoming DataFrame has new columns.")
    print("This is expected when Delta checks the target table schema and schema evolution is not enabled.")
    print(str(e))

StatementMeta(, 4faee51e-54ad-44c2-ae6c-7482b450a9ef, 7, Finished, Available, Finished, False)

Expected append issue because incoming DataFrame has new columns.
This is expected when Delta checks the target table schema and schema evolution is not enabled.
[_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 78152e2a-9d18-4700-884b-4b489857c737).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- customer_id: string (nullable = true)
-- customer_name: string (nullable = true)
-- email: string (nullable = true)
-- status: string (nullable = true)
-- customer_score: integer (nullable = true)
-- updated_at: timestamp (nullable = true)
-- source_system: string (nullable = true)


Data schema:
root
-- customer_id: string (nullable = true)
-- customer_name: string (nullable = true)

### Example 4: Controlled schema evolution with `mergeSchema`

ถ้า strict append fail และเราตัดสินใจแล้วว่า `phone_number` กับ `loyalty_tier` เป็น optional columns ที่รับได้ เราค่อยใช้ `mergeSchema=true`

สำคัญ: ใช้เฉพาะตอนที่ตั้งใจ evolve schema ไม่ใช่เปิดทิ้งไว้ทุก write แบบไม่ตรวจสอบ

In [6]:
if not append_without_merge_succeeded:
    (
        df_customer_v2_new_columns.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(target_table_name)
    )
    print("Appended batch v2 with controlled schema evolution.")
else:
    print("Skipped mergeSchema append to avoid duplicate rows.")

spark.read.table(target_table_name).printSchema()
spark.read.table(target_table_name).orderBy("customer_id").show(truncate=False)

StatementMeta(, 4faee51e-54ad-44c2-ae6c-7482b450a9ef, 8, Finished, Available, Finished, False)

Appended batch v2 with controlled schema evolution.
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- status: string (nullable = true)
 |-- customer_score: integer (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)

+-----------+-------------+-------------------+--------+--------------+-------------------+-------------+------------+------------+
|customer_id|customer_name|email              |status  |customer_score|updated_at         |source_system|phone_number|loyalty_tier|
+-----------+-------------+-------------------+--------+--------------+-------------------+-------------+------------+------------+
|C001       |Alice Wong   |alice@example.com  |active  |85            |2026-05-01 09:00:00|crm          |NULL        |NULL        |
|C002       |Boon Mee     

### Example 5: Compare incoming schema with target schema

ก่อน append batch ใหม่ เราควรมีวิธีตรวจว่า incoming schema ต่างจาก target schema ตรงไหน

Function ด้านล่างใช้สำหรับ learning lab เพื่อดู 3 เรื่องหลัก:

- new columns
- missing columns
- type mismatch

In [7]:
def schema_to_type_map(schema: T.StructType) -> dict:
    # Convert Spark StructType schema into a Python dictionary.
    column_type_map = {}

    for field in schema.fields:
        column_name = field.name  # Get column name from StructField.
        column_type = field.dataType.simpleString()  # Convert StructField dataType to a readable string, such as StringType() -> "string".

        # Add one dictionary entry by assigning value to key.
        # Pattern: dict[key] = value, for example "customer_id": "string".
        column_type_map[column_name] = column_type

    return column_type_map


def compare_schemas(expected_schema: T.StructType, incoming_schema: T.StructType):
    # Compare expected target schema with incoming DataFrame schema.

    # This function accepts schemas instead of DataFrames.
    # Schema drift detection only needs metadata: column names and data types.
    # It does not need to scan actual data rows.

    # This function returns a small DataFrame that shows new columns, missing columns, and type mismatches.
    
    expected = schema_to_type_map(expected_schema)
    incoming = schema_to_type_map(incoming_schema)

    rows = []  # Store schema drift result rows before converting to Spark DataFrame.

    expected_columns = set(expected.keys())
    incoming_columns = set(incoming.keys())

    # Columns that exist in incoming DataFrame but not in target table.
    new_columns = sorted(incoming_columns - expected_columns)

    for column_name in new_columns:
        # Add one result row as a tuple: (drift_type, column_name, expected_type, incoming_type).
        # A tuple is used because each row has a fixed column order that matches drift_schema.
        rows.append((
            "new_column",
            column_name,
            None,  # No expected type because this column does not exist in target schema.
            incoming[column_name],
        ))

    # Columns that exist in target table but not in incoming DataFrame.
    missing_columns = sorted(expected_columns - incoming_columns)

    for column_name in missing_columns:
        rows.append((
            "missing_column",
            column_name,
            expected[column_name],
            None,  # No incoming type because this column does not exist in incoming schema.
        ))

    # Columns that exist in both schemas.
    common_columns = sorted(expected_columns & incoming_columns)  # Use set intersection (&) to keep only values that exist in both sets.

    for column_name in common_columns:
        expected_type = expected[column_name]
        incoming_type = incoming[column_name]

        if expected_type != incoming_type:
            rows.append((
                "type_mismatch",
                column_name,
                expected_type,
                incoming_type,
            ))

    drift_schema = T.StructType([
        T.StructField("drift_type", T.StringType(), False),
        T.StructField("column_name", T.StringType(), False),
        T.StructField("expected_type", T.StringType(), True),
        T.StructField("incoming_type", T.StringType(), True),
    ])

    return spark.createDataFrame(rows, drift_schema)


target_schema_after_v2 = spark.read.table(target_table_name).schema

df_drift_check_v2 = compare_schemas(target_schema_after_v2, df_customer_v2_new_columns.schema)
df_drift_check_v2.show(truncate=False)

StatementMeta(, 4faee51e-54ad-44c2-ae6c-7482b450a9ef, 9, Finished, Available, Finished, False)

+----------+-----------+-------------+-------------+
|drift_type|column_name|expected_type|incoming_type|
+----------+-----------+-------------+-------------+
+----------+-----------+-------------+-------------+



### Example 6: Missing columns from source

Batch ถัดมาไม่มี `email` และ `loyalty_tier`

นี่คือ schema drift แบบ **missing columns**

ถ้า columns เหล่านี้เป็น optional เราอาจเติม `null` แล้ว align schema ให้ตรง target ก่อน append แต่ถ้าเป็น required field ควร fail หรือแยกเข้า quarantine/reject ในวันถัด ๆ ไป

In [8]:
customer_schema_v3_missing_columns = T.StructType([
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("customer_score", T.IntegerType(), True),
    T.StructField("updated_at", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("phone_number", T.StringType(), True),
])

customer_batch_v3_data = [
    ("C006", "Fah Sai", "active", 80, "2026-05-03 11:00:00", "crm_mobile", "0833333333"),
    ("C007", "Gawin Noon", "inactive", 55, "2026-05-03 11:05:00", "crm_mobile", "0844444444"),
]

df_customer_v3_missing_columns = (
    spark.createDataFrame(customer_batch_v3_data, customer_schema_v3_missing_columns)
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

compare_schemas(target_schema_after_v2, df_customer_v3_missing_columns.schema).show(truncate=False)

StatementMeta(, 4faee51e-54ad-44c2-ae6c-7482b450a9ef, 10, Finished, Available, Finished, False)

+--------------+------------+-------------+-------------+
|drift_type    |column_name |expected_type|incoming_type|
+--------------+------------+-------------+-------------+
|missing_column|email       |string       |NULL         |
|missing_column|loyalty_tier|string       |NULL         |
+--------------+------------+-------------+-------------+



### Example 7: Align missing columns before append

Function นี้ทำ 2 อย่าง:

1. ถ้า incoming ไม่มี target column ให้เติม column นั้นเป็น `null` และ cast type ให้ตรง target
2. ถ้า incoming มี column อยู่แล้ว ให้ cast เป็น type ตาม target schema

จากนั้น select columns ตาม target order เพื่อให้ write ต่อได้ง่ายขึ้น

Note:

- `mergeSchema` ใช้ไม่ได้กับ case นี้ (missing columns) เพราะ `mergeSchema` ใช้สำหรับเพิ่ม column ใหม่ที่ incoming DataFrame มี แต่ target table ยังไม่มี

- แต่กลับกัน case นี้ คือ target table มี column อยู่แล้ว แต่ incoming DataFrame ส่งมาไม่ครบ ดังนั้นต้อง align incoming DataFrame ให้ครบตาม target schema ก่อน append

In [9]:
def align_to_target_schema(df, target_schema: T.StructType):
    # Align a DataFrame to the target schema by adding missing columns and casting existing columns.
    aligned_df = df

    for field in target_schema.fields:
        if field.name not in aligned_df.columns:
            aligned_df = aligned_df.withColumn(field.name, F.lit(None).cast(field.dataType))
        else:
            aligned_df = aligned_df.withColumn(field.name, F.col(field.name).cast(field.dataType))

    # Select columns in target schema order so the output DataFrame follows the target table structure.
    return aligned_df.select([field.name for field in target_schema.fields])


# Tip:
# - df.withColumn("column_name", <Column expression>)
#   - Column expression can create a constant value, transform an existing column, cast a data type, or apply business logic.
# - List comprehension pattern: [output_value for item in collection]
#   - collection is the list or iterable we loop through.
#   - item is each element from the collection.
#   - output_value is what we want to keep in the new list.

df_customer_v3_aligned = align_to_target_schema(df_customer_v3_missing_columns, target_schema_after_v2)

df_customer_v3_aligned.printSchema()
df_customer_v3_aligned.show(truncate=False)

StatementMeta(, 4faee51e-54ad-44c2-ae6c-7482b450a9ef, 11, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = false)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- status: string (nullable = true)
 |-- customer_score: integer (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)

+-----------+-------------+-----+--------+--------------+-------------------+-------------+------------+------------+
|customer_id|customer_name|email|status  |customer_score|updated_at         |source_system|phone_number|loyalty_tier|
+-----------+-------------+-----+--------+--------------+-------------------+-------------+------------+------------+
|C006       |Fah Sai      |NULL |active  |80            |2026-05-03 11:00:00|crm_mobile   |0833333333  |NULL        |
|C007       |Gawin Noon   |NULL |inactive|55            |2026-05-03 11:05:00|crm_mobile   |0844444444  |NULL        |
+-----------+--

### Example 8: Append aligned data

เมื่อ schema align แล้ว เราสามารถ append เข้า target table ได้โดยไม่ต้อง evolve schema เพิ่ม

In [10]:
(
    df_customer_v3_aligned.write
    .format("delta")
    .mode("append")
    .saveAsTable(target_table_name)
)

spark.read.table(target_table_name).orderBy("customer_id").show(truncate=False)

StatementMeta(, 4faee51e-54ad-44c2-ae6c-7482b450a9ef, 12, Finished, Available, Finished, False)

+-----------+-------------+-------------------+--------+--------------+-------------------+-------------+------------+------------+
|customer_id|customer_name|email              |status  |customer_score|updated_at         |source_system|phone_number|loyalty_tier|
+-----------+-------------+-------------------+--------+--------------+-------------------+-------------+------------+------------+
|C001       |Alice Wong   |alice@example.com  |active  |85            |2026-05-01 09:00:00|crm          |NULL        |NULL        |
|C002       |Boon Mee     |boon@example.com   |active  |72            |2026-05-01 09:10:00|crm          |NULL        |NULL        |
|C003       |Chai Lee     |chai@example.com   |inactive|60            |2026-05-01 09:20:00|crm          |NULL        |NULL        |
|C004       |Dao Jai      |dao@example.com    |active  |91            |2026-05-02 10:00:00|crm          |0811111111  |gold        |
|C005       |Ekkarat Tan  |ekkarat@example.com|active  |77            |2026-

### Example 9: Type mismatch from source

Batch ถัดมา `customer_score` ถูกส่งมาเป็น string แทน integer และมีค่าที่ cast เป็น int ไม่ได้ เช่น `not_available`

นี่คือ schema drift แบบ **type mismatch**

การ cast ตรง ๆ อาจทำให้เกิด null หรือ error แล้วแต่ runtime/config ดังนั้นใน learning lab นี้เราจะ validate ด้วย regex ก่อน cast

In [11]:
customer_schema_v4_type_drift = T.StructType([
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("email", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("customer_score", T.StringType(), True),
    T.StructField("updated_at", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("phone_number", T.StringType(), True),
    T.StructField("loyalty_tier", T.StringType(), True),
])

customer_batch_v4_data = [
    ("C008", "Hana Mint", "hana@example.com", "active", "88", "2026-05-04 12:00:00", "crm", "0855555555", "gold"),
    ("C009", "Itim Dee", "itim@example.com", "active", "not_available", "2026-05-04 12:05:00", "crm", "0866666666", "silver"),
]

df_customer_v4_type_drift_raw = (
    spark.createDataFrame(customer_batch_v4_data, customer_schema_v4_type_drift)
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

compare_schemas(target_schema_after_v2, df_customer_v4_type_drift_raw.schema).show(truncate=False)

StatementMeta(, 4faee51e-54ad-44c2-ae6c-7482b450a9ef, 13, Finished, Available, Finished, False)

+-------------+--------------+-------------+-------------+
|drift_type   |column_name   |expected_type|incoming_type|
+-------------+--------------+-------------+-------------+
|type_mismatch|customer_score|int          |string       |
+-------------+--------------+-------------+-------------+



### Example 10: Cast type safely and flag invalid records

เราจะเก็บ raw value ไว้ใน `customer_score_raw` ชั่วคราว แล้วสร้าง `schema_issue` เพื่อ flag records ที่ cast ไม่ได้

Pattern นี้ช่วยให้ไม่ drop bad records เงียบ ๆ

In [18]:
df_customer_v4_type_controlled = (
    df_customer_v4_type_drift_raw
    .withColumn("customer_score_raw", F.col("customer_score"))
    .withColumn(
        "customer_score",
        F.when(F.col("customer_score").rlike(r"^-?\d+$"), F.col("customer_score").cast("int"))
         .otherwise(F.lit(None).cast("int"))
    )
    .withColumn(
        "schema_issue",
        # Non-matching records return null by default.
        # Optional: Use otherwise(F.lit(None).cast(...)) to make the null branch and data type explicit.
        F.when(
            F.col("customer_score_raw").isNotNull() & F.col("customer_score").isNull(),
            F.lit("invalid_customer_score")
        )
    )
)

# Tip:
# r"^-?\d+$" means optional negative sign followed by one or more digits, such as "100", "0", or "-5".

df_customer_v4_type_controlled.select(
    "customer_id",
    "customer_score_raw",
    "customer_score",
    "schema_issue"
).show(truncate=False)

StatementMeta(, 4faee51e-54ad-44c2-ae6c-7482b450a9ef, 20, Finished, Available, Finished, False)

+-----------+------------------+--------------+----------------------+
|customer_id|customer_score_raw|customer_score|schema_issue          |
+-----------+------------------+--------------+----------------------+
|C008       |88                |88            |NULL                  |
|C009       |not_available     |NULL          |invalid_customer_score|
+-----------+------------------+--------------+----------------------+



### Example 11: Append only valid records after type control

ใน production จริง invalid records ควรไป quarantine/reject table พร้อม error reason

สำหรับ Day 16 เราจะยังไม่ลงลึก quarantine pattern แต่จะ show ให้เห็นว่า valid records เท่านั้นที่ถูก append เข้า target

In [13]:
df_customer_v4_valid = (
    df_customer_v4_type_controlled
    .filter(F.col("schema_issue").isNull())
    .drop("customer_score_raw", "schema_issue")
)

df_customer_v4_invalid = (
    df_customer_v4_type_controlled
    .filter(F.col("schema_issue").isNotNull())
)

print("Valid records:")
df_customer_v4_valid.show(truncate=False)

print("Invalid records to review later:")
df_customer_v4_invalid.select(
    "customer_id",
    "customer_name",
    "customer_score_raw",
    "schema_issue"
).show(truncate=False)

# Align valid records to target schema before append.
df_customer_v4_valid_aligned = align_to_target_schema(df_customer_v4_valid, target_schema_after_v2)

(
    df_customer_v4_valid_aligned.write
    .format("delta")
    .mode("append")
    .saveAsTable(target_table_name)
)

spark.read.table(target_table_name).orderBy("customer_id").show(truncate=False)

StatementMeta(, 4faee51e-54ad-44c2-ae6c-7482b450a9ef, 15, Finished, Available, Finished, False)

Valid records:
+-----------+-------------+----------------+------+--------------+-------------------+-------------+------------+------------+
|customer_id|customer_name|email           |status|customer_score|updated_at         |source_system|phone_number|loyalty_tier|
+-----------+-------------+----------------+------+--------------+-------------------+-------------+------------+------------+
|C008       |Hana Mint    |hana@example.com|active|88            |2026-05-04 12:00:00|crm          |0855555555  |gold        |
+-----------+-------------+----------------+------+--------------+-------------------+-------------+------------+------------+

Invalid records to review later:
+-----------+-------------+------------------+----------------------+
|customer_id|customer_name|customer_score_raw|schema_issue          |
+-----------+-------------+------------------+----------------------+
|C009       |Itim Dee     |not_available     |invalid_customer_score|
+-----------+-------------+---------

### Example 12: Create a small schema drift log DataFrame

การ log schema drift ไม่จำเป็นต้องเริ่มจาก framework ใหญ่ ๆ เสมอไป

สำหรับ learning lab เราสร้าง DataFrame เล็ก ๆ เพื่อบันทึกว่าแต่ละ batch เจอ drift แบบไหน และจัดการอย่างไร

In [14]:
schema_drift_log_data = [
    ("batch_v2", "new_columns", "phone_number, loyalty_tier", "accepted_with_controlled_evolution"),
    ("batch_v3", "missing_columns", "email, loyalty_tier", "aligned_with_null_values"),
    ("batch_v4", "type_mismatch", "customer_score string -> int", "valid_records_appended_invalid_records_flagged"),
]

schema_drift_log_schema = T.StructType([
    T.StructField("batch_id", T.StringType(), False),
    T.StructField("drift_type", T.StringType(), False),
    T.StructField("detail", T.StringType(), True),
    T.StructField("handling_action", T.StringType(), True),
])

df_schema_drift_log = spark.createDataFrame(schema_drift_log_data, schema_drift_log_schema)

df_schema_drift_log.show(truncate=False)

StatementMeta(, 4faee51e-54ad-44c2-ae6c-7482b450a9ef, 16, Finished, Available, Finished, False)

+--------+---------------+----------------------------+----------------------------------------------+
|batch_id|drift_type     |detail                      |handling_action                               |
+--------+---------------+----------------------------+----------------------------------------------+
|batch_v2|new_columns    |phone_number, loyalty_tier  |accepted_with_controlled_evolution            |
|batch_v3|missing_columns|email, loyalty_tier         |aligned_with_null_values                      |
|batch_v4|type_mismatch  |customer_score string -> int|valid_records_appended_invalid_records_flagged|
+--------+---------------+----------------------------+----------------------------------------------+



## Quick recap

| Question | Answer |
|---|---|
| Schema drift คืออะไร? | Source schema เปลี่ยนไปจากที่ pipeline คาดไว้ |
| Schema evolution คืออะไร? | การเปลี่ยน target schema แบบตั้งใจและควบคุมได้ |
| New column ควรทำยังไง? | ตรวจว่าเป็น optional/approved ก่อนใช้ controlled evolution เช่น `mergeSchema` |
| Missing column ควรทำยังไง? | ถ้า optional อาจเติม null แล้ว align schema; ถ้า required ควร fail หรือ reject |
| Type mismatch ควรทำยังไง? | cast แบบระวัง, validate, และ flag/reject records ที่แปลงไม่ได้ |
| ก่อน append ควรเช็กอะไร? | schema difference, required columns, data type และ downstream impact |

## Exercises

### Exercise 1: Add a new optional column with controlled evolution

จำลอง source batch ใหม่ที่เพิ่ม column `preferred_language`

Requirements:

- สร้าง DataFrame ใหม่ที่มี column `preferred_language`
- ตรวจ schema drift เทียบกับ target table ปัจจุบัน
- append ด้วย `mergeSchema=true` เฉพาะเพราะ column นี้เป็น optional field ที่ยอมรับได้
- แสดง final schema หลัง append

Expected result:

- Target table มี column `preferred_language` เพิ่มขึ้น
- Row ใหม่ถูก append เข้า table

In [15]:
exercise_schema_new_optional_column = T.StructType([
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("email", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("customer_score", T.IntegerType(), True),
    T.StructField("updated_at", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("phone_number", T.StringType(), True),
    T.StructField("loyalty_tier", T.StringType(), True),
    T.StructField("preferred_language", T.StringType(), True),
])

exercise_new_optional_data = [
    ("C010", "Jinda Blue", "jinda@example.com", "active", 93, "2026-05-05 13:00:00", "crm", "0877777777", "gold", "th"),
]

df_exercise_new_optional = (
    spark.createDataFrame(exercise_new_optional_data, exercise_schema_new_optional_column)
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

current_target_schema = spark.read.table(target_table_name).schema
compare_schemas(current_target_schema, df_exercise_new_optional.schema).show(truncate=False)

(
    df_exercise_new_optional.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(target_table_name)
)

spark.read.table(target_table_name).printSchema()
spark.read.table(target_table_name).filter(F.col("customer_id") == "C010").show(truncate=False)

StatementMeta(, 4faee51e-54ad-44c2-ae6c-7482b450a9ef, 17, Finished, Available, Finished, False)

+----------+------------------+-------------+-------------+
|drift_type|column_name       |expected_type|incoming_type|
+----------+------------------+-------------+-------------+
|new_column|preferred_language|NULL         |string       |
+----------+------------------+-------------+-------------+

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- status: string (nullable = true)
 |-- customer_score: integer (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- preferred_language: string (nullable = true)

+-----------+-------------+-----------------+------+--------------+-------------------+-------------+------------+------------+------------------+
|customer_id|customer_name|email            |status|customer_score|updated_at         |source_system|phone_n

### Exercise 2: Align a batch with missing columns

จำลอง source batch ที่ไม่มี `email`, `loyalty_tier` และ `preferred_language`

Requirements:

- สร้าง incoming DataFrame ที่มีบาง columns หายไป
- ใช้ `compare_schemas()` เพื่อตรวจ missing columns
- ใช้ `align_to_target_schema()` เพื่อเติม missing columns เป็น null
- append เข้า target table

Expected result:

- DataFrame หลัง align มี schema ตรงกับ target table
- Row ใหม่ถูก append ได้โดยไม่ต้องใช้ `mergeSchema`

In [16]:
exercise_schema_missing_columns = T.StructType([
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("customer_score", T.IntegerType(), True),
    T.StructField("updated_at", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("phone_number", T.StringType(), True),
])

exercise_missing_data = [
    ("C011", "Kanda Dee", "active", 70, "2026-05-06 14:00:00", "crm_mobile", "0888888888"),
]

df_exercise_missing = (
    spark.createDataFrame(exercise_missing_data, exercise_schema_missing_columns)
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

current_target_schema = spark.read.table(target_table_name).schema
compare_schemas(current_target_schema, df_exercise_missing.schema).show(truncate=False)

df_exercise_missing_aligned = align_to_target_schema(df_exercise_missing, current_target_schema)
df_exercise_missing_aligned.printSchema()
df_exercise_missing_aligned.show(truncate=False)

(
    df_exercise_missing_aligned.write
    .format("delta")
    .mode("append")
    .saveAsTable(target_table_name)
)

spark.read.table(target_table_name).filter(F.col("customer_id") == "C011").show(truncate=False)

StatementMeta(, 4faee51e-54ad-44c2-ae6c-7482b450a9ef, 18, Finished, Available, Finished, False)

+--------------+------------------+-------------+-------------+
|drift_type    |column_name       |expected_type|incoming_type|
+--------------+------------------+-------------+-------------+
|missing_column|email             |string       |NULL         |
|missing_column|loyalty_tier      |string       |NULL         |
|missing_column|preferred_language|string       |NULL         |
+--------------+------------------+-------------+-------------+

root
 |-- customer_id: string (nullable = false)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- status: string (nullable = true)
 |-- customer_score: integer (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- preferred_language: string (nullable = true)

+-----------+-------------+-----+------+--------------+-------------------+-------------+------------+----

### Exercise 3: Detect type drift and keep invalid records visible

จำลอง source batch ที่ส่ง `customer_score` เป็น string และมีค่าที่แปลงเป็น int ไม่ได้

Requirements:

- สร้าง incoming DataFrame ที่ `customer_score` เป็น string
- ตรวจ type mismatch ด้วย `compare_schemas()`
- cast เฉพาะค่าที่เป็นตัวเลข
- แยก valid และ invalid records
- append เฉพาะ valid records เข้า target table

Expected result:

- Valid record ถูก append
- Invalid record ยังมองเห็นได้ใน DataFrame พร้อม `schema_issue`

In [22]:
exercise_schema_type_drift = T.StructType([
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("email", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("customer_score", T.StringType(), True),
    T.StructField("updated_at", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("phone_number", T.StringType(), True),
    T.StructField("loyalty_tier", T.StringType(), True),
    T.StructField("preferred_language", T.StringType(), True),
])

exercise_type_drift_data = [
    ("C012", "Lalida Sun", "lalida@example.com", "active", "66", "2026-05-07 15:00:00", "crm", "0899999999", "silver", "th"),
    ("C013", "Mali Moon", "mali@example.com", "active", "unknown", "2026-05-07 15:05:00", "crm", "0800000000", "bronze", "en"),
]

df_exercise_type_drift_raw = (
    spark.createDataFrame(exercise_type_drift_data, exercise_schema_type_drift)
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

current_target_schema = spark.read.table(target_table_name).schema
compare_schemas(current_target_schema, df_exercise_type_drift_raw.schema).show(truncate=False)

df_exercise_type_controlled = (
    df_exercise_type_drift_raw
    .withColumn("customer_score_raw", F.col("customer_score"))
    .withColumn(
        "customer_score",
        F.when(F.col("customer_score").rlike(r"^-?\d+$"), F.col("customer_score").cast("int"))
         .otherwise(F.lit(None).cast("int"))
    )
    .withColumn(
        "schema_issue",
        F.when(
            F.col("customer_score_raw").isNotNull() & F.col("customer_score").isNull(),
            F.lit("invalid_customer_score")
        )
    )
)

print("Invalid records should remain visible:")
df_exercise_type_controlled.filter(F.col("schema_issue").isNotNull()).select(
    "customer_id",
    "customer_score_raw",
    "schema_issue"
).show(truncate=False)

df_exercise_type_valid = (
    df_exercise_type_controlled
    .filter(F.col("schema_issue").isNull())
    .drop("customer_score_raw", "schema_issue")
)

df_exercise_type_valid_aligned = align_to_target_schema(df_exercise_type_valid, current_target_schema)

(
    df_exercise_type_valid_aligned.write
    .format("delta")
    .mode("append")
    .saveAsTable(target_table_name)
)

spark.read.table(target_table_name).filter(F.col("customer_id").isin("C012", "C013")).show(truncate=False)

StatementMeta(, 4faee51e-54ad-44c2-ae6c-7482b450a9ef, 24, Finished, Available, Finished, False)

+-------------+--------------+-------------+-------------+
|drift_type   |column_name   |expected_type|incoming_type|
+-------------+--------------+-------------+-------------+
|type_mismatch|customer_score|int          |string       |
+-------------+--------------+-------------+-------------+

Invalid records should remain visible:
+-----------+------------------+----------------------+
|customer_id|customer_score_raw|schema_issue          |
+-----------+------------------+----------------------+
|C013       |unknown           |invalid_customer_score|
+-----------+------------------+----------------------+

+-----------+-------------+------------------+------+--------------+-------------------+-------------+------------+------------+------------------+
|customer_id|customer_name|email             |status|customer_score|updated_at         |source_system|phone_number|loyalty_tier|preferred_language|
+-----------+-------------+------------------+------+--------------+-------------------+

## Common mistakes

1. **เปิด schema evolution แบบไม่คิด**

   ถ้าเปิดให้ table evolve ทุกครั้งโดยไม่ตรวจ schema ก่อน table อาจมี columns แปลก ๆ เพิ่มขึ้นเรื่อย ๆ และ downstream ควบคุมยาก

2. **คิดว่า new column ปลอดภัยเสมอ**

   New column อาจปลอดภัยถ้าเป็น optional field แต่ถ้าเป็น field สำคัญที่เปลี่ยน business logic ต้อง review ก่อน

3. **เติม missing column เป็น null โดยไม่ดู business rule**

   บาง column เป็น required field เช่น business key หรือ event time ถ้าหายไปไม่ควรเติม null แล้วปล่อยผ่านทันที

4. **cast type mismatch ตรง ๆ โดยไม่ตรวจ invalid values**

   ถ้า source ส่ง string เช่น `unknown` มาใน field ที่ควรเป็น int การ cast อาจกลายเป็น null หรือ error ต้อง flag ให้มองเห็น

5. **ไม่เก็บหลักฐานว่า schema เปลี่ยนเมื่อไร**

   ถ้า schema drift ทำให้ downstream พังภายหลัง แต่ไม่มี log หรือ note จะ debug ยากมาก

## Expected Output / Success Criteria

เมื่อจบ Day 16 ควรทำได้:

- อธิบายความต่างระหว่าง schema drift และ schema evolution ได้
- สร้าง initial Lakehouse table จาก DataFrame ได้
- จำลอง source batch ที่มี new columns ได้
- อธิบายได้ว่า append แบบ strict อาจ fail เมื่อ schema ไม่ตรง
- ใช้ `mergeSchema=true` แบบ controlled schema evolution ได้
- เขียน function เปรียบเทียบ target schema กับ incoming schema ได้
- ตรวจ new columns, missing columns และ type mismatch ได้
- เติม missing columns เป็น null และ align schema ก่อน append ได้
- handle type mismatch ด้วยการ validate/cast และ flag invalid records ได้
- อธิบายได้ว่า schema change ควรถูกตรวจและ document ก่อนเขียนลง target table

## Final takeaway

Schema drift เป็นเรื่องปกติของ real source system แต่การปล่อยให้ drift ไหลเข้า target table โดยไม่ตรวจคือความเสี่ยง

> ก่อนเขียนข้อมูลลง Lakehouse table ต้องรู้ว่า schema เปลี่ยนตรงไหน และตั้งใจเลือก response strategy เสมอ

Mindset ที่ควรจำ:

- New column ไม่ได้แปลว่าต้อง evolve ทันทีทุกครั้ง
- Missing column ต้องแยกว่า optional หรือ required
- Type mismatch ต้อง validate ก่อน cast
- Schema evolution ควรเป็น controlled action ไม่ใช่ side effect
- Invalid records ควรถูกมองเห็น ไม่ควรถูก drop เงียบ ๆ

## Optional cleanup

In [ ]:
# spark.sql(f"DROP TABLE IF EXISTS {target_table_name}")